In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import pandas as pd

In [2]:
from premise import *

In [3]:
start_time = time.time()

In [4]:
bd.projects.set_current('propanol_production_monte_carlo')

In [5]:
if 'ecoinvent-3.10-cutoff' not in bd.databases:
    bi.ecoinvent.import_ecoinvent_release('3.10', 'cutoff', username = 'SUPER_Lab', password = 'Ecoinvent?')

Applying strategy: normalize_units
Applying strategy: drop_unspecified_subcategories
Applying strategy: ensure_categories_are_tuples
Applied 3 strategies in 0.01 seconds
4362 datasets
	0 exchanges
	Links to the following databases:

	0 unlinked exchanges (0 types)
		


100%|██████████| 4362/4362 [00:00<00:00, 22571.11it/s]


21:53:20 [info     ] Vacuuming database            
Created database: ecoinvent-3.10-biosphere
Extracting XML data from 23523 datasets
Extracted 23523 datasets in 70.08 seconds
Applying strategy: normalize_units
Applying strategy: update_ecoinvent_locations
Applying strategy: remove_zero_amount_coproducts
Applying strategy: remove_zero_amount_inputs_with_no_activity
Applying strategy: remove_unnamed_parameters
Applying strategy: es2_assign_only_product_with_amount_as_reference_product
Applying strategy: assign_single_product_as_activity
Applying strategy: create_composite_code
Applying strategy: drop_unspecified_subcategories
Applying strategy: fix_ecoinvent_flows_pre35
Applying strategy: drop_temporary_outdated_biosphere_flows
Applying strategy: link_biosphere_by_flow_uuid
Applying strategy: link_internal_technosphere_by_composite_code
Applying strategy: delete_exchanges_missing_activity
Applying strategy: delete_ghost_exchanges
Applying strategy: remove_uncertainty_from_negative_loss

100%|██████████| 23523/23523 [01:01<00:00, 384.86it/s]


21:55:51 [info     ] Vacuuming database            
Created database: ecoinvent-3.10-cutoff


In [6]:
bd.databases

Databases dictionary with 2 object(s):
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff

In [7]:
# import LCIA implementation data
METHODS_DIR = Path(r'..\data\ecoinvent\lcia_implementation_3.10.xlsx')
PREMISE_GWP_DIR = Path(r'..\data\ecoinvent\lcia_gwp_2021_100a_w_bio.xlsx')

lcia_ei_310 = pd.read_excel(METHODS_DIR, sheet_name = 'CFs')
premise_gwp = pd.read_excel(PREMISE_GWP_DIR)

In [8]:
# get 100 year GWPs from IPCC 2021 as implemented in Ecoinvent v3.10
ipcc_2021_gwp_100a = lcia_ei_310[(lcia_ei_310['Method'] == 'IPCC 2021') &
                                 (lcia_ei_310['Category'] == 'climate change') &
                                 (lcia_ei_310['Indicator'] == 'global warming potential (GWP100)')]

# adapt GWPs from IPCC 2021 to premise GWPs
premise_ipcc_2021_gwp_100a = dict()
for index, row in premise_gwp.iterrows():
    premise_ipcc_2021_gwp_100a.update({(row['name'], row['categories']): row['amount']})

    # update existing and add new characterisation factors based on IPCC 2021
    for index, row in ipcc_2021_gwp_100a.iterrows():
        ef_info = (row['Name'], row['Compartment'] + '::' + row['Subcompartment'])
        # update existing elementary flows
        if ef_info in premise_ipcc_2021_gwp_100a:
            premise_ipcc_2021_gwp_100a[ef_info] = row['CF']
        else:
            premise_ipcc_2021_gwp_100a.update({ef_info: row['CF']})

    # export the modified IPCC method
    premise_ipcc_2021_df = pd.DataFrame(premise_ipcc_2021_gwp_100a, index = ['amount']).T.reset_index()
    premise_ipcc_2021_df.columns = ['name', 'categories', 'amount']
    premise_ipcc_2021_df.to_excel(Path(r'..\data\ecoinvent\lcia_gwp_100a_ipcc_2021_premise.xlsx'), index = False)

In [9]:
# write method:
metadata =(
           ("IPCC 2021", "climate change", "GWP 100a, incl. H and bio CO2"),
            "kg CO2-Eq",
            "IPCC 2021, with CFs for hydrogen and biogenic CO2 flows",
            Path(r'..\data\ecoinvent\lcia_gwp_100a_ipcc_2021_premise.xlsx'),
          )

ipcc_2021_method = bi.ExcelLCIAImporter(filepath = metadata[-1], name = metadata[0], unit = metadata[1], description = metadata[2])
ipcc_2021_method.apply_strategies()
ipcc_2021_method.drop_unlinked()
assert len(list(ipcc_2021_method.unlinked)) == 0
ipcc_2021_method.write_methods(overwrite = True, verbose = True)

Applying strategy: csv_restore_tuples
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: set_biosphere_type
Applying strategy: drop_unspecified_subcategories
Applying strategy: link_iterable_by_fields
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applied 8 strategies in 0.11 seconds
Applying strategy: drop_unlinked_cfs
Applied 1 strategies in 0.00 seconds
Wrote 1 LCIA methods with 209 characterization factors
